# เฉลยโจทย์ Bayesian Network: Smoking → Cancer → X-Ray

โครงสร้างเครือข่าย: **S (Smoker) → C (Cancer) → T (X-Ray Test)**

**หมายเหตุการแปลงตัวแปร:** ไลบรารีนี้รองรับเฉพาะตัวแปรแบบ True/False เราจึงกำหนดให้
`T = True` หมายถึง **"บวก" (positive)** และ `T = False` หมายถึง **"ลบ" (negative)**


## 1. คลาสหลักจาก `probability.py`

In [1]:
from collections import defaultdict

# ---------- ฟังก์ชันช่วยเหลือ (เทียบเท่ากับที่อยู่ใน utils.py ของ aima-python) ----------

def extend(s, var, val):
    '''คืนค่า dict s ที่เพิ่ม var: val เข้าไป (ไม่แก้ของเดิม)'''
    s2 = dict(s)
    s2[var] = val
    return s2

def event_values(event, variables):
    '''คืนค่า tuple ของค่าตัวแปรใน event ตามลำดับ variables'''
    return tuple([event[var] for var in variables])


# ---------- ProbDist, BayesNode, BayesNet (คัดลอก logic จาก probability.py) ----------

class ProbDist:
    '''Discrete probability distribution สำหรับตัวแปรสุ่มหนึ่งตัว'''
    def __init__(self, varname='?', freqs=None):
        self.prob = {}
        self.varname = varname
        self.values = []
        if freqs:
            for (v, p) in freqs.items():
                self[v] = p
            self.normalize()

    def __getitem__(self, val):
        return self.prob[val]

    def __setitem__(self, val, p):
        if val not in self.values:
            self.values.append(val)
        self.prob[val] = p

    def normalize(self):
        total = sum(self.prob.values())
        if not (1 - 1e-9 <= total <= 1 + 1e-9):
            for vk in self.prob:
                self.prob[vk] /= total
        return self

    def show_approx(self, numfmt='{:.3g}'):
        return ', '.join([('{}: ' + numfmt).format(v, p)
                           for (v, p) in sorted(self.prob.items())])


class BayesNode:
    '''Conditional Probability Table สำหรับตัวแปร boolean หนึ่งตัว P(X | parents)'''
    def __init__(self, X, parents, cpt):
        if isinstance(parents, str):
            parents = parents.split()
        if isinstance(cpt, (float, int)):
            cpt = {(): cpt}
        elif isinstance(cpt, dict):
            if cpt and isinstance(list(cpt.keys())[0], bool):
                cpt = {(v,): p for v, p in cpt.items()}
        self.variable = X
        self.parents = parents
        self.cpt = cpt
        self.children = []

    def p(self, value, event):
        '''P(X=value | parents=parent_values ตาม event)'''
        ptrue = self.cpt[event_values(event, self.parents)]
        return ptrue if value else 1 - ptrue

    def __repr__(self):
        return repr((self.variable, ' '.join(self.parents)))


class BayesNet:
    '''Bayesian Network ที่ประกอบด้วย BayesNode (ตัวแปร boolean เท่านั้น)'''
    def __init__(self, node_specs=None):
        self.nodes = []
        self.variables = []
        for node_spec in (node_specs or []):
            self.add(node_spec)

    def add(self, node_spec):
        node = BayesNode(*node_spec)
        assert node.variable not in self.variables
        assert all((parent in self.variables) for parent in node.parents)
        self.nodes.append(node)
        self.variables.append(node.variable)
        for parent in node.parents:
            self.variable_node(parent).children.append(node)

    def variable_node(self, var):
        for n in self.nodes:
            if n.variable == var:
                return n
        raise Exception("No such variable: {}".format(var))

    def variable_values(self, var):
        return [True, False]

    def __repr__(self):
        return 'BayesNet({0!r})'.format(self.nodes)


# ---------- Exact Inference: Enumeration (ตรงตาม Figure 14.9 ของ probability.py) ----------

def enumerate_all(variables, e, bn):
    '''ผลรวมของ P(variables | e) ที่สอดคล้องกับ evidence e'''
    if not variables:
        return 1.0
    Y, rest = variables[0], variables[1:]
    Ynode = bn.variable_node(Y)
    if Y in e:
        return Ynode.p(e[Y], e) * enumerate_all(rest, e, bn)
    else:
        return sum(Ynode.p(y, e) * enumerate_all(rest, extend(e, Y, y), bn)
                   for y in bn.variable_values(Y))

def enumeration_ask(X, e, bn):
    '''คืนค่า P(X | e) แบบ normalized ProbDist'''
    assert X not in e, "Query variable must be distinct from evidence"
    Q = ProbDist(X)
    for xi in bn.variable_values(X):
        Q[xi] = enumerate_all(bn.variables, extend(e, X, xi), bn)
    return Q.normalize()

print("โหลดคลาส BayesNode / BayesNet / enumeration_ask เรียบร้อย")


โหลดคลาส BayesNode / BayesNet / enumeration_ask เรียบร้อย


## 2. สร้างเครือข่าย S → C → T ตามข้อมูลในโจทย์

| ตาราง | ความหมาย |
|---|---|
| Table 1 | Prior P(S) |
| Table 2 | P(C \| S) |
| Table 3 | P(T \| C)  — T=True แทน "บวก", T=False แทน "ลบ" |


In [2]:
T, F = True, False

net = BayesNet([
    ('S', '',    0.30),                                   # P(S=true) = 0.30
    ('C', 'S',   {T: 0.05, F: 0.01}),                      # P(C=true|S=true)=0.05, P(C=true|S=false)=0.01
    ('T', 'C',   {T: 0.90, F: 0.20}),                      # P(T=บวก|C=true)=0.90, P(T=บวก|C=false)=0.20
])

def th(v):
    '''แปลค่า boolean กลับเป็นข้อความไทยสำหรับแสดงผล'''
    return v

print(net)


BayesNet([('S', ''), ('C', 'S'), ('T', 'C')])


---
## ข้อ 1: หาค่า P(S=true, C=true, T="บวก")

ใช้ chain rule ของเครือข่าย (joint probability):

$$P(S,C,T) = P(S)\cdot P(C\mid S)\cdot P(T\mid C)$$


In [3]:
S_node = net.variable_node('S')
C_node = net.variable_node('C')
T_node = net.variable_node('T')

pS  = S_node.p(True, {})
pC  = C_node.p(True, {'S': True})
pT  = T_node.p(True, {'C': True})

joint = pS * pC * pT
print(f"P(S=true)            = {pS}")
print(f"P(C=true | S=true)   = {pC}")
print(f"P(T=บวก  | C=true)   = {pT}")
print(f"P(S=true, C=true, T=บวก) = {pS} x {pC} x {pT} = {joint:.4f}")
print("\nคำตอบ: b) 0.0135" if abs(joint-0.0135)<1e-9 else f"\n=> {joint}")


P(S=true)            = 0.3
P(C=true | S=true)   = 0.05
P(T=บวก  | C=true)   = 0.9
P(S=true, C=true, T=บวก) = 0.3 x 0.05 x 0.9 = 0.0135

คำตอบ: b) 0.0135


---
## ข้อ 42: หาโอกาสทั้งหมดที่จะให้ผลตรวจเอกซเรย์เป็นบวก P(T="บวก")

Marginalize ตัวแปร S และ C ออก โดยใช้ `enumeration_ask` (ไม่ต้องคำนวณมือ)


In [4]:
dist_T = enumeration_ask('T', {}, net)
p_T_positive = dist_T[True]

print("P(T) =", dist_T.show_approx())
print(f"\nP(T=บวก) = {p_T_positive:.4f}")
print("\nคำตอบ: b) 0.2154" if abs(p_T_positive-0.2154)<1e-3 else f"\n=> {p_T_positive}")


P(T) = False: 0.785, True: 0.215

P(T=บวก) = 0.2154

คำตอบ: b) 0.2154


---
## ข้อ 43: Posterior — P(C=true | T="บวก")

ใช้กฎของ Bayes ผ่าน `enumeration_ask` โดยกำหนด evidence T=True


In [5]:
dist_C_given_T = enumeration_ask('C', {'T': True}, net)
p_C_given_T = dist_C_given_T[True]

print("P(C | T=บวก) =", dist_C_given_T.show_approx())
print(f"\nP(C=true | T=บวก) = {p_C_given_T:.4f}")
print("\nคำตอบ: b) 0.092" if abs(p_C_given_T-0.092)<1e-3 else f"\n=> {p_C_given_T}")


P(C | T=บวก) = False: 0.908, True: 0.0919

P(C=true | T=บวก) = 0.0919

คำตอบ: b) 0.092


---
## ข้อ 44: Posterior — P(S=true | T="บวก")

ตัวแปร S ไม่ใช่พ่อแม่โดยตรงของ T แต่ `enumeration_ask` จะ marginalize ตัวแปรแฝง (C) ให้อัตโนมัติ


In [6]:
dist_S_given_T = enumeration_ask('S', {'T': True}, net)
p_S_given_T = dist_S_given_T[True]

print("P(S | T=บวก) =", dist_S_given_T.show_approx())
print(f"\nP(S=true | T=บวก) = {p_S_given_T:.4f}")
print("\nคำตอบ: a) 0.327" if abs(p_S_given_T-0.327)<1e-3 else f"\n=> {p_S_given_T}")


P(S | T=บวก) = False: 0.673, True: 0.327

P(S=true | T=บวก) = 0.3273

คำตอบ: a) 0.327


---
## ข้อ 45: จาก 1,000 คน คาดว่าจะเป็นมะเร็ง (C=true) **และ** ผลเอกซเรย์บวกกี่คน

$$P(C=true,\;T=บวก) = P(C=true)\cdot P(T=บวก\mid C=true)$$

แล้วคูณด้วยจำนวนผู้ป่วยทั้งหมด (1,000 คน)


In [7]:
dist_C = enumeration_ask('C', {}, net)
p_C_true = dist_C[True]
p_C_and_T = p_C_true * T_node.p(True, {'C': True})

n_patients = 1000
expected_count = n_patients * p_C_and_T

print(f"P(C=true)               = {p_C_true:.4f}")
print(f"P(T=บวก | C=true)       = {T_node.p(True, {'C': True})}")
print(f"P(C=true, T=บวก)        = {p_C_and_T:.4f}")
print(f"\nจำนวนคนคาดการณ์ (จาก 1000 คน) = {expected_count:.1f} คน  ≈ {round(expected_count)} คน")
print("\nคำตอบ: b) ≈ 20" if round(expected_count) == 20 else f"\n=> ≈ {round(expected_count)}")


P(C=true)               = 0.0220
P(T=บวก | C=true)       = 0.9
P(C=true, T=บวก)        = 0.0198

จำนวนคนคาดการณ์ (จาก 1000 คน) = 19.8 คน  ≈ 20 คน

คำตอบ: b) ≈ 20


---
## สรุปคำตอบทั้งหมด

| ข้อ | คำถาม | คำตอบ |
|---|---|---|
| 1  | P(S=true, C=true, T="บวก") | **b) 0.0135** |
| 42 | P(T="บวก") | **b) 0.2154** |
| 43 | P(C=true \| T="บวก") | **b) 0.092** |
| 44 | P(S=true \| T="บวก") | **a) 0.327** |
| 45 | จำนวนคน (มะเร็ง+ผลบวก) จาก 1000 คน | **b) ≈ 20** |
